In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import os
import pathlib
import random as rn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import argparse
import argparse
import json
import tensorboard
import tensorboardX
import os
import argparse
import json
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim 
import nni
from nni.nas.nn.pytorch import ModelSpace, LayerChoice, MutableConv2d, MutableBatchNorm2d, MutableReLU
from pytorch_lightning import Trainer
from nni.nas.evaluator.pytorch import Lightning, ClassificationModule, Trainer
from nni.nas.experiment import NasExperiment
from nni.nas.space import model_context
from nni.nas.hub.pytorch import DARTS
from nni.nas.strategy import DARTS as DartsStrategy
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchvision import transforms
from torchvision.datasets import CIFAR10
from nni.nas.experiment import NasExperiment
from nni.nas.evaluator import FunctionalEvaluator
from nni.nas.evaluator import FunctionalEvaluator
import nni.nas.strategy as strategy
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
import genotypes
from pytorch_lightning.callbacks import ModelCheckpoint
torch.set_float32_matmul_precision('medium')
from tqdm import tqdm
from nni.nas.nn.pytorch import LayerChoice, ModelSpace,ValueChoice
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from pytorch_lightning import LightningModule, Trainer
from torchvision import datasets, transforms
from nni.nas.evaluator.pytorch import Classification

In [2]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# Transforms for MNIST (32x32 resize, then normalization)
mnist_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize to 32x32 if needed
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize(mean=(0.5,), std=(0.5,))  # Normalize for single channel (grayscale)
])

# Download and create datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=mnist_transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=mnist_transform)

# Split train dataset into training and validation sets (80% train, 20% validation)
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=12)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True, num_workers=12)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=12)

# Get a batch of images and labels from the training loader
data_iter = iter(train_loader)
images, labels = next(data_iter)

# Print the shapes of the images and labels
print(f"Images shape: {images.shape}")  # Should be [16, 1, 32, 32] for batch size 16 and 1 channel
print(f"Labels shape: {labels.shape}")  # Should be [16] for 16 labels
print(f"Labels: {labels}")  # Print the actual labels


Images shape: torch.Size([16, 1, 32, 32])
Labels shape: torch.Size([16])
Labels: tensor([0, 1, 1, 2, 6, 2, 2, 2, 1, 8, 2, 6, 2, 6, 8, 1])


In [3]:
@nni.trace
class AuxLossClassificationModule(ClassificationModule):
    """Several customization for the training of DARTS, based on default Classification."""
    model: DARTS
    def __init__(self,
                 learning_rate: float = 0.025,
                 weight_decay: float = 0.,
                 auxiliary_loss_weight: float = 0.4,
                 max_epochs: int = 600):
        print(f"lr : {learning_rate}")
        print(f"weight decay: {weight_decay}")
        print(f"aux loss weight: {auxiliary_loss_weight}")
        print(f"max epochs: {max_epochs}")
        super().__init__(learning_rate=learning_rate, weight_decay=weight_decay, num_classes=10)
        self.auxiliary_loss_weight = auxiliary_loss_weight
        self.max_epochs = max_epochs
        self.criterion=  nn.CrossEntropyLoss()
        self.l1_coeff = 1e-5

    def configure_optimizers(self):
        """Customized optimizer with momentum, as well as a scheduler."""
        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=1e-5, 
            betas=(0.9, 0.999), 
            eps=1e-07,
            weight_decay=self.auxiliary_loss_weight
        )

        # Define the scheduler here (Cosine Annealing)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, 
            T_max=self.max_epochs,  # Number of iterations to perform the annealing
            eta_min=1e-7  # Final learning rate after annealing
        )

        # Return optimizer and scheduler in the form of a dictionary
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'train_loss',  # You can monitor a specific metric
                'interval': 'epoch',  # When to step the scheduler ('epoch' or 'step')
                'frequency': 1  # How often to update the learning rate
            }
        }

    def training_step(self, batch, batch_idx):
        """Training step, customized with auxiliary loss."""
        x, y = batch

        # Check for NaNs or infinite values in input
        if torch.isnan(x).any() or torch.isnan(y).any() or torch.isinf(x).any() or torch.isinf(y).any():
            raise ValueError("Input data contains NaNs or Infinities.")
        y_hat = self(x)
        loss_main = self.criterion(y_hat, y)
        l1_regularization = sum(p.abs().sum() for p in self.parameters())
        loss = loss_main + self.l1_coeff * l1_regularization      
        #acc = (y_hat.argmax(dim=1) == y).float().mean()
        self.log('train_loss', loss, prog_bar=True)
        #self.log('train_accuracy', acc, prog_bar=True)  # Log training accuracy
        for name, metric in self.metrics.items():
            self.log('train_' + name, metric(y_hat, y), prog_bar=True)
        return loss
        



    def on_train_epoch_start(self):
        """Set drop path probability before every epoch. This has no effect if drop path is not enabled in model."""
        self.model.set_drop_path_prob(self.model.drop_path_prob * self.current_epoch / self.max_epochs)

        # Logging learning rate at the beginning of every epoch
        self.log('lr', self.trainer.optimizers[0].param_groups[0]['lr'])




In [4]:

class MnistModelSpace(ModelSpace):
    def __init__(self, input_channels, channels, num_classes, layers, verbose):
        super(MnistModelSpace, self).__init__()
        #self.first_iter = True
        self.layers = nn.ModuleList()
        self.drop_path_prob = 0.0  
        self.preliminary_layer = nn.Conv2d(1, 8, kernel_size=3, padding=0, bias=False)
        self.verbose = verbose
        layer1 = LayerChoice([
              nn.Conv2d(8, 16, kernel_size=1, padding=0, bias=False),   
              nn.Conv2d(8, 16, kernel_size=5, padding=2, bias=False)
        ], label='layer_1')
        self.layers.append(layer1)
        self.bn = nn.BatchNorm2d(16)

        # Ensure the number of inputs to fc1 is close to but not exceeding 200
        self.pool = nn.AdaptiveAvgPool2d((2, 2))
        self.fc1 = nn.Linear(16 * 2 * 2, 128) 
        self.relu = nn.ReLU()
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        # Kernel: 3
        # Initial shape: 32x32
        #print(f'Input shape: {x.shape}')
        x = self.preliminary_layer(x)
        # After preliminary layer: 31x31
        if self.verbose == 1 :
            print(f'After preliminary layer: {x.shape}')
        for i, layer in enumerate(self.layers):
            x = layer(x)
        x = self.bn(x)
        x= self.relu(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        if self.verbose == 1 :
            print(f'After flattening: {x.shape}')
        
        x = self.fc1(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc1: {x.shape}')
        x = self.classifier(x)
        if self.verbose == 1 :
            print(f'After classifier: {x.shape}')
        #self.first_iter = False
        return x

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob
        for layer in self.layers:
            if hasattr(layer, 'set_drop_path_prob'):
                layer.set_drop_path_prob(drop_path_prob)


In [5]:
def search(log_dir: str, batch_size: int = 256):
    """
    Darts search 

    Args:
        log_dir (str): The directory where logs will be saved.
        batch_size (int, optional): The size of the batches. Default is 64.  
    Returns:
        None
    """

    model_space =MnistModelSpace(input_channels=1, channels=64, num_classes=10, layers=1,verbose =0)
    model_space.set_drop_path_prob(0.2)

    checkpoint_callback = ModelCheckpoint(
        monitor='train_acc',  
        dirpath='./checkpoints', 
        filename='best-checkpoint',  
        save_top_k=1,
        mode='max'  
    )


    evaluator = Lightning(
        AuxLossClassificationModule(1e-5, 3e-4, 0., 100),
        Trainer(
            accelerator="auto",
            callbacks=[checkpoint_callback],  
            max_epochs=100
        ),
        train_dataloaders=train_loader,
        val_dataloaders=val_loader
    )

    strategy = nni.nas.strategy.DARTS()

    experiment = NasExperiment(model_space, evaluator, strategy)
    experiment.run()
    return experiment



In [6]:
experiment_results = search("./",256)

lr : 1e-05
weight decay: 0.0003
aux loss weight: 0.0
max epochs: 100


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


[2024-10-18 03:59:17] Config is not provided. Will try to infer.
[2024-10-18 03:59:17] Strategy is found to be a one-shot strategy. Setting execution engine to "sequential" and format to "raw".
[2024-10-18 03:59:17] WARNING: `training_service` will be ignored for sequential execution engine.
[2024-10-18 03:59:17] WARNING: `training_service` will be ignored for sequential execution engine.
[2024-10-18 03:59:17] WARNING: `training_service` will be ignored for sequential execution engine.
[2024-10-18 03:59:17] WARNING: `training_service` will be ignored for sequential execution engine.
[2024-10-18 03:59:17] WARNING: `training_service` will be ignored for sequential execution engine.
[2024-10-18 03:59:17] WARNING: `training_service` will be ignored for sequential execution engine.
[2024-10-18 03:59:17] WARNING: `training_service` will be ignored for sequential execution engine.
[2024-10-18 03:59:17] WARNING: `training_service` will be ignored for sequential execution engine.
[2024-10-18 03

C:\Users\senti\anaconda3\envs\NNI_NAS_local\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:653: Checkpoint directory C:\Users\senti\Documents\GitHub\PhotonicNas\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\senti\anaconda3\envs\NNI_NAS_local\Lib\site-packages\pytorch_lightning\core\optimizer.py:315: The lr scheduler dict contains the key(s) ['monitor'], but the keys will be ignored. You need to call `lr_scheduler.step()` manually in manual optimization.

  | Name            | Type                        | Params
----------------------------------------------------------------
0 | training_module | AuxLossClassificationModule | 13.0 K
----------------------------------------------------------------
13.0 K    Trainable params
0         Non-trainable params
13.0 K    Total params
0.052     Total estimated model params size (MB)
C:\Users\senti\anaconda3\envs\NNI_NAS_local\Lib\site-packages\pytorch_lightning\trainer\connectors\dat

Epoch 12:  26%|██▌       | 770/3000 [01:01<02:57, 12.60it/s, v_num=56, train_loss=1.070, train_acc=0.812] 

...

Epoch 85:  50%|█████     | 1500/3000 [01:17<01:17, 19.40it/s, v_num=56, train_loss=0.146, train_acc=1.000] 

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Epoch 99: 100%|██████████| 3000/3000 [01:42<00:00, 29.39it/s, v_num=56, train_loss=0.149, train_acc=1.000] 

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 3000/3000 [01:42<00:00, 29.39it/s, v_num=56, train_loss=0.149, train_acc=1.000]
[2024-10-18 07:00:35] Waiting for models submitted to engine to finish...
[2024-10-18 07:00:35] Experiment is completed.
[2024-10-18 07:00:35] WARNING: `training_service` will be ignored for sequential execution engine.
